# 2 — Bidirectional transformers for regression

**Learning goal:** predict one continuous number from a variable-length numeric
sequence using a transformer encoder **without a causal mask**.

Unlike next-token generation, every observed timestep is available when estimating
the target. Earlier representations may therefore attend to later observations.
We will create a dataset with known structure, beat a simple baseline, inspect
errors, and directly test bidirectional information flow.


## Problem and pipeline

Imagine each row is a short sensor history with two features: a noisy signal and a
control variable. The target depends on the signal's global average, its trend, and
an interaction. Sequences have different lengths.

`generate → split → fit train-only scaler → Dataset/DataLoader → pad + mask →
numeric projection + positions → bidirectional encoder → masked mean → regression
head → MSE → optimize → MAE/RMSE/R² + residual plots`

This synthetic task is scientifically useful: the true data-generating process is
known, so a bug cannot hide behind mysterious real-world data.


In [ ]:
from pathlib import Path
import math, random
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torch.utils.tensorboard import SummaryWriter

SEED = 7
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): device = torch.device("cuda")
elif torch.backends.mps.is_available(): device = torch.device("mps")
else: device = torch.device("cpu")
FAST_MODE = True
print(torch.__version__, device)


### Python bridge

`range(n)` produces 0 through n-1. A tuple such as `(x, y)` groups values. `*batch`
unpacks a tuple into function arguments (similar in spirit to R's `do.call`). A
leading underscore in `_private` is only a convention. `None` is Python's missing
object value; unlike numeric `NaN`, it has no arithmetic meaning.


## 1. Generate and visualize data

Each example has length 12–40 and shape `(T, 2)`. The target is continuous, so this
is **regression**. Classification chooses categories; language modeling performs a
classification at every token position.

The target formula is never provided to the model. It must approximate the mapping
from examples. Added noise creates irreducible error: even a perfect model should
not achieve exactly zero test MSE.


In [ ]:
def make_example(rng, min_len=12, max_len=40):
    length = int(rng.integers(min_len, max_len + 1))
    t = np.linspace(0, 1, length, dtype=np.float32)
    level = rng.normal(0, 1); slope = rng.normal(0, .8)
    control = rng.uniform(-1, 1, size=length).astype(np.float32)
    signal = (level + slope*t + .35*np.sin(2*np.pi*t) + rng.normal(0,.12,length)).astype(np.float32)
    x = np.stack([signal, control], axis=1)
    y = 1.8*signal.mean() + 1.2*(signal[-1]-signal[0]) + .7*(signal*control).mean() + rng.normal(0,.12)
    return torch.tensor(x), torch.tensor(y, dtype=torch.float32)

rng = np.random.default_rng(SEED)
examples = [make_example(rng) for _ in range(500 if FAST_MODE else 2000)]
print("examples:", len(examples), "first x/y shapes:", examples[0][0].shape, examples[0][1].shape)
for x, y in examples[:4]: plt.plot(x[:,0], alpha=.8, label=f"y={y:.2f}")
plt.xlabel("timestep"); plt.ylabel("signal"); plt.legend(); plt.show()


### Exercise — split by example

Use `random_split` for 70% train, 15% validation, and the remainder test. Pass a seeded `torch.Generator` so the split is reproducible.

Replace the `None`/`TODO` portion below. The next cell is a small unit test: green
output means the behavior and important shapes are correct, not that there is only
one valid solution.


In [ ]:
train_set = None
val_set = None
test_set = None


In [ ]:
if train_set is None:
    print("🟡 Not attempted. API: random_split(examples, [n_train,n_val,n_test], generator=...).")
else:
    assert len(train_set)+len(val_set)+len(test_set) == len(examples)
    assert set(train_set.indices).isdisjoint(test_set.indices)
    print("🟢 Split sizes:", len(train_set), len(val_set), len(test_set))


## 2. Preprocessing without leakage

Features measured in different units can make optimization difficult. Standardize
each feature as `(x - train_mean) / train_std`. Fit these statistics on **training
data only**. Using validation/test statistics leaks information from the future
evaluation distribution into training.

Targets can also be standardized; this keeps early gradients in a convenient range.
Convert predictions back to original units before reporting metrics.


### Exercise — fit train-only scaling statistics

Concatenate timesteps from training examples, then compute feature-wise mean/std and scalar target mean/std. Use population standard deviation (`correction=0`).

Replace the `None`/`TODO` portion below. The next cell is a small unit test: green
output means the behavior and important shapes are correct, not that there is only
one valid solution.


In [ ]:
# Use the reference indices only if your split exercise is unfinished.
_gen = torch.Generator().manual_seed(SEED)
_train, _val, _test = random_split(examples, [350,75,75], generator=_gen)
feature_mean = None; feature_std = None; target_mean = None; target_std = None


In [ ]:
if feature_mean is None:
    print("🟡 Not attempted. Hint: torch.cat([x for x,y in _train], dim=0).")
else:
    assert feature_mean.shape == feature_std.shape == (2,)
    assert torch.all(feature_std > 0) and target_std > 0
    print("🟢 Feature mean/std:", feature_mean, feature_std)


## 3. Padding, masks, and DataLoader

A minibatch tensor must be rectangular, but sequence lengths differ. We pad shorter
examples with zeros to the longest sequence in that batch and create a Boolean
**padding mask** shaped `(B,T)`. `True` means “ignore this padded position.”

This mask is not causal. It hides nonexistent padding but leaves all real past and
future positions mutually visible. Dynamic per-batch padding wastes less work than
padding every example to the dataset maximum.

`Dataset` defines how to retrieve one item; `DataLoader` batches, optionally
shuffles, and calls `collate_fn` to combine variable-size items.

Reference: [`DataLoader`](https://docs.pytorch.org/docs/stable/data.html),
[`pad_sequence`](https://docs.pytorch.org/docs/stable/generated/torch.nn.utils.rnn.pad_sequence.html)


### Exercise — collate variable-length sequences

Write a collate function returning padded `x`, targets `y`, and a Boolean padding mask. You may use `pad_sequence(..., batch_first=True)`.

Replace the `None`/`TODO` portion below. The next cell is a small unit test: green
output means the behavior and important shapes are correct, not that there is only
one valid solution.


In [ ]:
def collate_batch(batch):
    # batch is a list of (x, y) tuples
    return None, None, None


In [ ]:
px, py, pmask = collate_batch(examples[:4])
if px is None:
    print("🟡 Not attempted. Build lengths, then compare arange(max_len) >= lengths[:,None].")
else:
    assert px.ndim == 3 and py.shape == (4,) and pmask.shape == px.shape[:2]
    assert pmask.dtype == torch.bool
    assert torch.all(px[pmask] == 0)
    print("🟢 Shapes x/y/mask:", px.shape, py.shape, pmask.shape)


## 4. Encoder regression architecture

Numeric vectors do not use a token lookup table. A linear **input projection** maps
2 features to `d_model` learned features. Learned positional vectors add order.
Transformer blocks then mix information across all real timesteps.

To predict one value per sequence, **masked mean pooling** averages only real output
positions. A regression head maps the pooled vector to one scalar. Alternatives
include a special `[CLS]` token, max pooling, or attention pooling.

There is no causal attention mask. We pass only `src_key_padding_mask`.

Reference: [`TransformerEncoderLayer`](https://docs.pytorch.org/docs/stable/generated/torch.nn.TransformerEncoderLayer.html),
[`MSELoss`](https://docs.pytorch.org/docs/stable/generated/torch.nn.MSELoss.html)


### Exercise — define bidirectional regression

Complete input projection, positions, encoder, masked mean, and scalar head. Return shape `(B,)`.

Replace the `None`/`TODO` portion below. The next cell is a small unit test: green
output means the behavior and important shapes are correct, not that there is only
one valid solution.


In [ ]:
class SequenceRegressor(nn.Module):
    def __init__(self, n_features=2, d_model=64, n_head=4, n_layer=2, max_len=40):
        super().__init__()
        # TODO

    def forward(self, x, padding_mask):
        # TODO — importantly, no causal mask
        return None


In [ ]:
try:
    reg = SequenceRegressor().to(device)
    pred = reg(px.to(device), pmask.to(device))
    assert pred.shape == (4,) and torch.isfinite(pred).all()
    print("🟢 Prediction shape:", pred.shape)
except Exception as exc:
    print("🟡 Finish the model, then rerun:", type(exc).__name__, exc)


## 5. Baselines and regression metrics

Always compare with a cheap baseline. “Predict the training target mean” has no
features and often exposes broken pipelines. Metrics in original target units:

- **MSE:** mean squared error; emphasizes large mistakes and is smooth to optimize.
- **RMSE:** square root of MSE; same units as target.
- **MAE:** mean absolute error; less dominated by outliers.
- **R²:** improvement over predicting the evaluation-set mean; 1 is perfect, 0 is
  no better, and negative is worse. It is not a percentage of predictions correct.

Optimize standardized-target MSE, but report all metrics after inverse scaling.


### Exercise — implement metrics

Given equal one-dimensional tensors `pred` and `actual`, return a dictionary containing mse, rmse, mae, and r2.

Replace the `None`/`TODO` portion below. The next cell is a small unit test: green
output means the behavior and important shapes are correct, not that there is only
one valid solution.


In [ ]:
def regression_metrics(pred, actual):
    return None


In [ ]:
m = regression_metrics(torch.tensor([1.,2.,3.]), torch.tensor([1.,2.,4.]))
if m is None:
    print("🟡 Not attempted.")
else:
    assert abs(m["mse"] - 1/3) < 1e-6 and 0 <= m["r2"] <= 1
    print("🟢", m)


## 6. Training, early stopping, and diagnosis

An **epoch** is one pass through the training loader. Shuffle training examples each
epoch, but not validation/test. Early stopping saves the checkpoint with lowest
validation loss and stops after `patience` unimproved epochs. Test remains untouched.

TensorBoard records loss and learning rate. Residual plots later show whether errors
are centered randomly or reveal bias/nonlinearity/unequal variance.


# Answer key — complete runnable implementation

The independent `ans_` pipeline below includes splitting, scaling, collating,
training, checkpointing, test metrics, plots, and a direct masking experiment.


In [ ]:
# Answers 1–3: split, train-only preprocessing, and collation
g = torch.Generator().manual_seed(SEED)
n_train = int(.70*len(examples)); n_val = int(.15*len(examples))
ans_train, ans_val, ans_test = random_split(examples, [n_train,n_val,len(examples)-n_train-n_val], generator=g)
train_x = torch.cat([x for x,y in ans_train], dim=0)
train_y = torch.stack([y for x,y in ans_train])
ans_xmean = train_x.mean(0); ans_xstd = train_x.std(0, correction=0).clamp_min(1e-6)
ans_ymean = train_y.mean(); ans_ystd = train_y.std(correction=0).clamp_min(1e-6)

def ans_collate(batch):
    xs, ys = zip(*batch)
    lengths = torch.tensor([len(x) for x in xs])
    padded = nn.utils.rnn.pad_sequence(xs, batch_first=True)
    mask = torch.arange(padded.size(1))[None,:] >= lengths[:,None]
    padded = (padded - ans_xmean) / ans_xstd
    targets = (torch.stack(ys) - ans_ymean) / ans_ystd
    return padded, targets, mask

train_loader = DataLoader(ans_train, batch_size=32, shuffle=True, collate_fn=ans_collate)
val_loader = DataLoader(ans_val, batch_size=64, collate_fn=ans_collate)
test_loader = DataLoader(ans_test, batch_size=64, collate_fn=ans_collate)


In [ ]:
# Answer 4: bidirectional encoder and masked pooling
class AnswerSequenceRegressor(nn.Module):
    def __init__(self, n_features=2, d_model=64, n_head=4, n_layer=2, max_len=40):
        super().__init__()
        self.input_projection = nn.Linear(n_features, d_model)
        self.position_embedding = nn.Embedding(max_len, d_model)
        layer = nn.TransformerEncoderLayer(d_model, n_head, 4*d_model, .1,
                                           activation="gelu", batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(layer, n_layer, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Sequential(nn.Linear(d_model, d_model//2), nn.GELU(), nn.Linear(d_model//2, 1))

    def encode(self, x, padding_mask):
        pos = torch.arange(x.size(1), device=x.device)
        h = self.input_projection(x) + self.position_embedding(pos)[None,:,:]
        return self.encoder(h, src_key_padding_mask=padding_mask)  # no causal mask

    def forward(self, x, padding_mask):
        h = self.norm(self.encode(x, padding_mask))
        valid = (~padding_mask).unsqueeze(-1).to(h.dtype)
        pooled = (h * valid).sum(1) / valid.sum(1).clamp_min(1)
        return self.head(pooled).squeeze(-1)

ans_model = AnswerSequenceRegressor().to(device)
print(f"Parameters: {sum(p.numel() for p in ans_model.parameters()):,}")


In [ ]:
# Answer 5: metric implementation, collection, and mean baseline
def ans_metrics(pred, actual):
    error = pred - actual
    mse = (error**2).mean()
    return {"mse": mse.item(), "rmse": mse.sqrt().item(),
            "mae": error.abs().mean().item(),
            "r2": (1 - (error**2).sum()/((actual-actual.mean())**2).sum()).item()}

test_targets = torch.stack([y for x,y in ans_test])
baseline = torch.full_like(test_targets, ans_ymean)
print("Mean baseline:", ans_metrics(baseline, test_targets))


In [ ]:
# Answer 6: explicit epoch loop with early stopping
def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train(training); total = count = 0
    context = torch.enable_grad() if training else torch.no_grad()
    with context:
        for x, y, mask in loader:
            x, y, mask = x.to(device), y.to(device), mask.to(device)
            pred = model(x, mask); loss = F.mse_loss(pred, y)
            if training:
                optimizer.zero_grad(set_to_none=True); loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); optimizer.step()
            total += loss.item()*len(y); count += len(y)
    return total/count

epochs = 10 if FAST_MODE else 60; patience = 5
optimizer = torch.optim.AdamW(ans_model.parameters(), lr=1e-3, weight_decay=1e-3)
writer = SummaryWriter("runs/regression"); Path("checkpoints").mkdir(exist_ok=True)
history = {"train":[], "val":[]}; best = float("inf"); stale = 0
for epoch in range(epochs):
    tr = run_epoch(ans_model, train_loader, optimizer); va = run_epoch(ans_model, val_loader)
    history["train"].append(tr); history["val"].append(va)
    writer.add_scalars("standardized_mse", {"train":tr, "validation":va}, epoch)
    print(f"epoch {epoch+1:02d} | train {tr:.4f} | val {va:.4f}")
    if va < best:
        best=va; stale=0; torch.save(ans_model.state_dict(), "checkpoints/regression_best.pt")
    else:
        stale += 1
        if stale >= patience: print("early stopping"); break
writer.close()


In [ ]:
# Final evaluation in original units and diagnostic plots
@torch.no_grad()
def collect_predictions(model, loader):
    model.eval(); ps=[]; ys=[]
    for x,y,mask in loader:
        p = model(x.to(device), mask.to(device)).cpu()
        ps.append(p*ans_ystd+ans_ymean); ys.append(y*ans_ystd+ans_ymean)
    return torch.cat(ps), torch.cat(ys)

ans_model.load_state_dict(torch.load("checkpoints/regression_best.pt", map_location=device, weights_only=True))
pred, actual = collect_predictions(ans_model, test_loader)
print("Transformer:", ans_metrics(pred, actual))
fig, ax = plt.subplots(1,3,figsize=(13,3.5))
ax[0].plot(history["train"],label="train"); ax[0].plot(history["val"],label="validation"); ax[0].set_title("Learning curves"); ax[0].legend()
ax[1].scatter(actual,pred,alpha=.7); lo=min(actual.min(),pred.min()); hi=max(actual.max(),pred.max()); ax[1].plot([lo,hi],[lo,hi],"k--"); ax[1].set(xlabel="actual",ylabel="predicted",title="Predictions")
ax[2].scatter(pred,pred-actual,alpha=.7); ax[2].axhline(0,color="k",ls="--"); ax[2].set(xlabel="predicted",ylabel="residual",title="Residuals")
plt.tight_layout(); plt.show()


## 7. Prove the mask distinction

This is the conceptual heart of the notebook. Change only the final real timestep
and compare the encoded representation at position 0. With bidirectional attention,
position 0 can change because it sees the future. With a correct causal mask it
would remain unchanged (in evaluation mode) because future positions are forbidden.

The padding mask still matters: changing a *padded* value should not affect pooled
predictions when masking is implemented correctly.


In [ ]:
ans_model.eval()
x, y, mask = next(iter(test_loader)); x=x[:1].to(device); mask=mask[:1].to(device)
x_changed = x.clone(); last_real = int((~mask[0]).sum())-1; x_changed[0,last_real,0] += 5
with torch.no_grad():
    h1 = ans_model.encode(x,mask); h2 = ans_model.encode(x_changed,mask)
delta = (h1[0,0]-h2[0,0]).abs().mean().item()
print(f"Mean change at position 0 after changing a future token: {delta:.6f}")
assert delta > 1e-6


In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs/regression


## Interpret and iterate

If the transformer does not beat the mean baseline, first verify inverse target
scaling, masks, and that optimizer updates occur. Then increase epochs. Training
loss far below validation suggests overfitting; add data, dropout, weight decay, or
reduce capacity. Structured residual curves suggest missing model flexibility or a
preprocessing issue.

Experiments:

1. Remove the trend term from data generation and compare difficulty.
2. Replace masked mean pooling with the final real timestep.
3. Add a causal mask and measure the result; is it intrinsically wrong, or merely
   an unnecessary restriction for this sequence-to-one task?
4. Compare a linear model or MLP on hand-engineered mean/trend features.
5. Increase noise to observe irreducible error.

Real regression tasks also require missing-data policies, distribution shift checks,
domain-informed splits, uncertainty, and fairness/safety evaluation.


## References

- PyTorch [`random_split`](https://docs.pytorch.org/docs/stable/data.html#torch.utils.data.random_split)
- PyTorch [`TransformerEncoder`](https://docs.pytorch.org/docs/stable/generated/torch.nn.TransformerEncoder.html)
- PyTorch [`TransformerEncoderLayer.forward`](https://docs.pytorch.org/docs/stable/generated/torch.nn.TransformerEncoderLayer.html#torch.nn.TransformerEncoderLayer.forward)
- PyTorch [`MSELoss`](https://docs.pytorch.org/docs/stable/generated/torch.nn.MSELoss.html)
- PyTorch [TensorBoard recipe](https://docs.pytorch.org/tutorials/recipes/recipes/tensorboard_with_pytorch.html)

**Completion check:** explain why the scaler is train-only, how padding and causal
masks differ, why pooling is needed, what a negative R² means, and why the test set
is evaluated once.
